In [1]:
from Lists.abbreviations import INTERNET_SLANG
from Lists.emoticons import EMOTICONS
from Lists.emojis import EMO_UNICODE
import pandas as pd
import contractions
import unicodedata
import re

In [3]:
df = pd.read_csv('./OLID/OLID.csv')

In [4]:
df = df[~((df['label_A'] == 'OFF') & (df['label_B'] == 'TIN') & (df['label_C'].isna()))]

label_dictionary = {
    ('NOT', None, None): 'Not Offensive',
    ('OFF', 'UNT', None): 'Untargeted Offense',
    ('OFF', 'TIN', 'IND'): 'Targeted Offense - Individual',
    ('OFF', 'TIN', 'GRP'): 'Targeted Offense - Group',
    ('OFF', 'TIN', 'OTH'): 'Targeted Offense - Other'
}


def map_type(row):
    key = (row['label_A'], row['label_B'], row['label_C'])
    return label_dictionary.get(key, "Unknown")

df[['label_A', 'label_B', 'label_C']] = df[['label_A', 'label_B', 'label_C']].replace({pd.NA: None, 'nan': None, 'None': None})

df['type'] = df.apply(map_type, axis=1)

In [5]:
type_labels = {label: idx for idx, label in enumerate(df['label_A'].unique())}
print(type_labels)
df['label'] = df['label_A'].map(type_labels)

{'NOT': 0, 'OFF': 1}


In [6]:
df.head()

,text,label_A,label_B,label_C,type,label
0,Has @USER quit? I've not heard of any #knifecr...,NOT,None,None,Not Offensive,0
1,"In celebration of Emancipation Day, we urge yo...",NOT,None,None,Not Offensive,0
2,@USER @USER It’d be a literal dream come true ...,NOT,None,None,Not Offensive,0
3,Brilliant news to read that Hoggy has signed a...,NOT,None,None,Not Offensive,0
4,@USER She speaks of the truth 😌,NOT,None,None,Not Offensive,0


In [7]:
UNICODE_EMO = {v: k for k, v in EMO_UNICODE.items()}


def convert_emoticons(text):
    for emot in EMOTICONS:
        text = re.sub(u'(' + emot + ')', "  ".join(EMOTICONS[emot].replace(",", "").split()), text)
    return text


def convert_emojis(text):
    for emot in UNICODE_EMO:
        text = re.sub(r'(' + emot + ')', "  ".join(UNICODE_EMO[emot].replace(",", "").replace(":", "").split()), text)
    return text


def convert_abbreviations(text):
    for abbr in INTERNET_SLANG:
        pattern = re.compile(r'\b' + re.escape(abbr) + r'\b', re.IGNORECASE)
        replacement = " ".join(INTERNET_SLANG[abbr].replace(",", "").split())
        text = pattern.sub(replacement, text)
    return text


def remove_unknown_emojis(text):
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002702-\U000027B0"  # miscellaneous symbols
                               u"\U0001F900-\U0001F9FF"  # supplemental symbols
                               u"\U0001F200-\U0001F251"  # enclosed characters
                               u"\U0001F004-\U0001F0CF"  # playing cards
                               u"\U00002B50"  # star symbol
                               "]+", flags=re.UNICODE)

    emojis_in_text = emoji_pattern.findall(text)

    for emoji_char in emojis_in_text:
        if emoji_char not in UNICODE_EMO:
            text = text.replace(emoji_char, '')
    return text


def remove_accented_chars(text):
    text = contractions.fix(text)
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

In [8]:
def convert_emojis_and_slang(text):
    text = convert_emoticons(text)
    text = convert_emojis(text)
    text = remove_unknown_emojis(text)
    text = convert_abbreviations(text)
    text=remove_accented_chars(text)

    return text

In [9]:
from tqdm.notebook import tqdm

In [10]:
def preprocess_text(df_input):
    tqdm.pandas() 

    df_comments = df_input.copy()
    df_comments['text'] = df_comments['text'].str.replace(r"http\S+", "URL", regex=True)  # remove URLs
    df_comments['text'] = df_comments['text'].str.replace(r'@[A-Za-z0-9_.]+', 'USER', regex=True)  # remove user mentions
    df_comments['text'] = df_comments['text'].str.replace(r'#+(\S+)', r'\1', regex=True)  # remove hashtags
    df_comments['text'] = df_comments['text'].str.replace(r'\s+', ' ', regex=True)  # remove extra spaces
    df_comments['text'] = df_comments['text'].str.replace(r'\d+', '', regex=True)  # remove digits

    df_comments['text'] = df_comments['text'].progress_apply(lambda x: convert_emojis_and_slang(x))

    df_comments = df_comments.drop_duplicates()

    return df_comments

In [11]:
df = preprocess_text(df)

  0%|          | 0/5989 [00:00<?, ?it/s]

In [12]:
df.head()

,text,label_A,label_B,label_C,type,label
0,Has USER quit? I have not heard of any knifecr...,NOT,None,None,Not Offensive,0
1,"In celebration of Emancipation Day, we urge yo...",NOT,None,None,Not Offensive,0
2,USER USER It would be a literal dream come tru...,NOT,None,None,Not Offensive,0
3,Brilliant news to read that Hoggy has signed a...,NOT,None,None,Not Offensive,0
4,USER She speaks of the truth relieved_face,NOT,None,None,Not Offensive,0


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5790 entries, 0 to 5992
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     5790 non-null   object
 1   label_A  5790 non-null   object
 2   label_B  2892 non-null   object
 3   label_C  1456 non-null   object
 4   type     5790 non-null   object
 5   label    5790 non-null   int64 
dtypes: int64(1), object(5)
memory usage: 316.6+ KB


In [14]:
df.to_csv('./OLID_transformers.csv',index=False)